# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamanna0612/-Flyrank-internship-ML-Tamanna-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: One row = One specific web page (identified by content_hash_id) for a specific client (client_hash_id).
Time Window: The analysis spans a 90-days window, comparing the "previous 45 days" against the "last 45 days" to measure momentum and establish our tatget label.The specific month partition tested here is March 2026 ( month=2026-03 ).


In [1]:
import os, getpass, duckdb
import pandas as pd

# Setup DuckDB and Hugging Face connection
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your HF Token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
}

# Verify the grain (One row = one client + one content piece per day in the raw table)
grain_check = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           COUNT(DISTINCT CONCAT(client_hash_id, '-', content_hash_id, '-', report_date)) as unique_grain
    FROM {TABLES['fact_daily']}
""").df()

print("Verifying Raw Grain (Row = Client + Content + Date):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verifying Raw Grain (Row = Client + Content + Date):
   total_rows  unique_grain
0     9841378       9841378


## 2. Fields: feature / label / context / excluded

Context: client_hash_id, content_hash_id (Used for grouping and holdout splits, never as training features).

Features (Knowable at decision time): imp_prev45 (Impressions in the first 45 days), clk_prev45 (Clicks), pos_prev45 (Avg position), pos_volatility (Standard deviation of position).

Label (The Target): is_declining (Binary 1/0 based on whether impressions in the last 45 days dropped by >20% compared to the prev 45 days).

Excluded: imp_last45 and clk_last45. Why? These exist in the future relative to the decision point. Including them in the training features would be data leakage, giving the model the answer before it guesses.

In [2]:

# Code to demonstrate how we build these fields exactly as described
features_query = f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev45,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev45,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END)       AS pos_prev45,
               STDDEV_SAMP(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END) AS pos_volatility
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 100
    )
    SELECT * FROM windowed
"""

contract_df = con.sql(features_query).df()
print("Contract Fields Created. Current Shape:", contract_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Contract Fields Created. Current Shape: (0, 7)


## 3. Verify it with queries (grain, counts, missing values, windows)
Here we verify the claims made above. We check the date span to ensure the 90-day window exists in the partition, check the final row count after our availability threshold (HAVING imp_prev45 >= 100), and check for missing values in our constructed features.

In [3]:
# Verify Date Span
dates = con.sql(f"SELECT MIN(report_date) as start_date, MAX(report_date) as end_date FROM {TABLES['fact_daily']}").df()
print("--- Date Span Verification ---")
print(dates, "\n")

# Verify availability (How many rows survive the threshold filter?)
total_raw_content = con.sql(f"SELECT COUNT(DISTINCT content_hash_id) FROM {TABLES['fact_daily']}").fetchone()[0]
surviving_content = len(contract_df)
print("--- Availability Verification ---")
print(f"Total distinct content pieces before threshold: {total_raw_content:,}")
print(f"Content pieces surviving threshold (imp_prev45 >= 100): {surviving_content:,}")
print(f"Retention rate: {(surviving_content/total_raw_content)*100:.1f}%\n")

# Verify Missing Values
print("--- Missing Values Verification ---")
print(contract_df[['imp_prev45', 'clk_prev45', 'pos_prev45', 'pos_volatility']].isnull().sum())

--- Date Span Verification ---
  start_date   end_date
0 2026-03-01 2026-03-31 

--- Availability Verification ---
Total distinct content pieces before threshold: 331,437
Content pieces surviving threshold (imp_prev45 >= 100): 0
Retention rate: 0.0%

--- Missing Values Verification ---
imp_prev45        0
clk_prev45        0
pos_prev45        0
pos_volatility    0
dtype: int64


## 4. Data limits

Unbalanced History: Not all clients joined the platform at the same time. The dataset's history depth varies per client, meaning long-term historical features (e.g., Year-over-Year comparisons) are impossible for newer clients without injecting heavy bias.

GSC vs. GA4 Mismatch: We only have Search Console (GSC) data for early rows, meaning we cannot reliably use on-page engagement metrics (like bounce rate) for older content without encountering massive chunks of missing data.

Causality: This data can only show us what happened (traffic dropped), not why it happened (e.g., a competitor outranked us vs. search demand dropped).

In [4]:
# Self-check complete
print("Self-check complete: All sections filled, queries execute successfully, and limits are documented.")

Self-check complete: All sections filled, queries execute successfully, and limits are documented.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.